In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import time

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
X_train = np.load("X_train_20step.npy", mmap_mode="r")
y_train = np.load("y_train_20step.npy", mmap_mode="r")

X_val = np.load("X_val_20step.npy", mmap_mode="r")
y_val = np.load("y_val_20step.npy", mmap_mode="r")

X_test = np.load("X_test_20step.npy", mmap_mode="r")
y_test = np.load("y_test_20step.npy", mmap_mode="r")

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (44325, 20, 105) (44325,)
Validation: (9655, 20, 105) (9655,)
Test: (10075, 20, 105) (10075,)


In [3]:
X_train_static = X_train[:, -1, :]
X_val_static = X_val[:, -1, :]
X_test_static = X_test[:, -1, :]

print("Static train:", X_train_static.shape)
print("Static validation:", X_val_static.shape)
print("Static test:", X_test_static.shape)

Static train: (44325, 105)
Static validation: (9655, 105)
Static test: (10075, 105)


In [4]:
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    r2 = r2_score(y_true, y_pred)

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [5]:
train_mean_rul = y_train.mean()

print("Mean training RUL:", train_mean_rul)

mean_val_pred = np.full(
    len(y_val),
    train_mean_rul
)

mean_test_pred = np.full(
    len(y_test),
    train_mean_rul
)

Mean training RUL: 88.34149


In [6]:
mean_val_result = evaluate_model(
    "Mean RUL",
    y_val,
    mean_val_pred
)

mean_test_result = evaluate_model(
    "Mean RUL",
    y_test,
    mean_test_pred
)

print("Validation:")
print(mean_val_result)

print("\nTest:")
print(mean_test_result)

Validation:
{'Model': 'Mean RUL', 'MAE': 55.59067916870117, 'RMSE': np.float64(68.74065277366807), 'R2': -0.0003014802932739258}

Test:
{'Model': 'Mean RUL', 'MAE': 55.3885498046875, 'RMSE': np.float64(69.17588966495119), 'R2': -5.173683166503906e-05}


In [7]:
print("Training Linear Regression...")

start_time = time.time()

linear_model = LinearRegression()

linear_model.fit(
    X_train_static,
    y_train
)

elapsed = time.time() - start_time

print(
    f"Linear Regression training time: {elapsed:.2f} seconds"
)

Training Linear Regression...
Linear Regression training time: 0.87 seconds


In [8]:
linear_val_pred = linear_model.predict(
    X_val_static
)

linear_test_pred = linear_model.predict(
    X_test_static
)

print("Predictions generated.")

Predictions generated.


In [9]:
linear_val_result = evaluate_model(
    "Linear Regression",
    y_val,
    linear_val_pred
)

linear_test_result = evaluate_model(
    "Linear Regression",
    y_test,
    linear_test_pred
)

print("Validation:")
print(linear_val_result)

print("\nTest:")
print(linear_test_result)

Validation:
{'Model': 'Linear Regression', 'MAE': 49.041648864746094, 'RMSE': np.float64(64.1820873143843), 'R2': 0.12797021865844727}

Test:
{'Model': 'Linear Regression', 'MAE': 51.820552825927734, 'RMSE': np.float64(68.6913883681026), 'R2': 0.013907790184020996}


In [10]:
print("Training Random Forest...")

start_time = time.time()

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    max_features="sqrt"
)

rf_model.fit(
    X_train_static,
    y_train
)

elapsed = time.time() - start_time

print(
    f"Random Forest training time: {elapsed:.2f} seconds"
)

Training Random Forest...
Random Forest training time: 13.86 seconds


In [11]:
rf_val_pred = rf_model.predict(
    X_val_static
)

rf_test_pred = rf_model.predict(
    X_test_static
)

print("Predictions generated.")

Predictions generated.


In [12]:
rf_val_result = evaluate_model(
    "Random Forest",
    y_val,
    rf_val_pred
)

rf_test_result = evaluate_model(
    "Random Forest",
    y_test,
    rf_test_pred
)

print("Validation:")
print(rf_val_result)

print("\nTest:")
print(rf_test_result)

Validation:
{'Model': 'Random Forest', 'MAE': 43.799820008686034, 'RMSE': np.float64(56.21181244381224), 'R2': 0.33110364336924536}

Test:
{'Model': 'Random Forest', 'MAE': 43.201120833072984, 'RMSE': np.float64(56.22973533510693), 'R2': 0.339238049351383}


In [13]:
validation_results = pd.DataFrame([
    mean_val_result,
    linear_val_result,
    rf_val_result
])

test_results = pd.DataFrame([
    mean_test_result,
    linear_test_result,
    rf_test_result
])

print("===== VALIDATION RESULTS =====")
display(validation_results)

print("\n===== TEST RESULTS =====")
display(test_results)

===== VALIDATION RESULTS =====


,Model,MAE,RMSE,R2
0,Mean RUL,55.590679,68.740653,-0.000301
1,Linear Regression,49.041649,64.182087,0.127970
2,Random Forest,43.799820,56.211812,0.331104



===== TEST RESULTS =====


,Model,MAE,RMSE,R2
0,Mean RUL,55.388550,69.175890,-0.000052
1,Linear Regression,51.820553,68.691388,0.013908
2,Random Forest,43.201121,56.229735,0.339238


In [14]:
test_results_display = test_results.copy()

test_results_display["MAE"] = (
    test_results_display["MAE"].round(4)
)

test_results_display["RMSE"] = (
    test_results_display["RMSE"].round(4)
)

test_results_display["R2"] = (
    test_results_display["R2"].round(4)
)

display(test_results_display)

,Model,MAE,RMSE,R2
0,Mean RUL,55.3885,69.1759,-0.0001
1,Linear Regression,51.8206,68.6914,0.0139
2,Random Forest,43.2011,56.2297,0.3392


In [15]:
validation_results.to_csv(
    "baseline_validation_results.csv",
    index=False
)

test_results.to_csv(
    "baseline_test_results.csv",
    index=False
)

print("Saved:")
print("baseline_validation_results.csv")
print("baseline_test_results.csv")

Saved:
baseline_validation_results.csv
baseline_test_results.csv


In [16]:
import joblib

joblib.dump(
    linear_model,
    "linear_regression_baseline.pkl"
)

joblib.dump(
    rf_model,
    "random_forest_baseline.pkl"
)

print("Baseline models saved.")


Baseline models saved.
